In [14]:
# %% [markdown]
# # HW14 – Эмбеддинги, FAISS, оценка retrieval и mini-RAG
# 
# **Тема:** Основы искусственного интеллекта (учебная база знаний)
# 
# Выполнены все обязательные пункты:
# - загрузка и анализ базы знаний
# - чанкинг документов
# - построение эмбеддингов (sentence-transformers)
# - индекс FAISS и поиск
# - контрольные запросы и метрики hit@k, recall@k
# - эксперимент с chunk_size
# - обновление базы знаний и переиндексация
# - mini-RAG (extractive генератор)
# - анализ ошибок
# 
# Артефакты сохранены в `artifacts/`.

# %% [code]
# 1. Импорты, seed, устройство
import os
import re
import random
import numpy as np
import pandas as pd
from typing import List, Dict
import matplotlib.pyplot as plt

# Для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Устройство для эмбеддингов
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

# %% [code]
# 2. База знаний
documents: List[Dict[str, str]] = [
    {"doc_id": "doc_01", "title": "Что такое ИИ?",
     "text": "Искусственный интеллект (ИИ) — это область компьютерных наук, занимающаяся созданием систем, способных выполнять задачи, требующие человеческого интеллекта: распознавание речи, принятие решений, перевод языков."},
    {"doc_id": "doc_02", "title": "Машинное обучение",
     "text": "Машинное обучение — подраздел ИИ, в котором алгоритмы обучаются на данных. Выделяют supervised learning (обучение с учителем), unsupervised learning (без учителя) и reinforcement learning (обучение с подкреплением)."},
    {"doc_id": "doc_03", "title": "Нейронные сети",
     "text": "Искусственные нейронные сети вдохновлены биологическими нейронами. Состоят из слоёв: входной, скрытые, выходной. Обучение происходит с помощью обратного распространения ошибки и градиентного спуска."},
    {"doc_id": "doc_04", "title": "Глубокое обучение",
     "text": "Глубокое обучение (deep learning) использует нейронные сети с большим числом скрытых слоёв. Особенно эффективно для обработки изображений (CNN) и последовательных данных (RNN, Transformer)."},
    {"doc_id": "doc_05", "title": "Обработка естественного языка (NLP)",
     "text": "NLP занимается взаимодействием компьютеров с человеческим языком. Задачи: токенизация, извлечение сущностей, машинный перевод, анализ тональности. Современные подходы используют трансформеры (BERT, GPT)."},
    {"doc_id": "doc_06", "title": "Компьютерное зрение",
     "text": "Компьютерное зрение позволяет машинам интерпретировать визуальную информацию. Ключевые задачи: классификация изображений, обнаружение объектов, сегментация. Свёрточные нейронные сети (CNN) — основа многих решений."},
    {"doc_id": "doc_07", "title": "Этика ИИ",
     "text": "Этические вопросы ИИ включают предвзятость алгоритмов, конфиденциальность данных, ответственность за решения, автоматизацию рабочих мест. Важно разрабатывать прозрачные и справедливые системы."},
    {"doc_id": "doc_08", "title": "Будущее ИИ",
     "text": "Ожидаемые направления: сильный ИИ (AGI), объяснимый ИИ (XAI), ИИ в здравоохранении, автономные системы. Потенциальные риски связаны с безопасностью и контролем."},
    {"doc_id": "doc_09", "title": "Инструменты ИИ",
     "text": "Популярные библиотеки: TensorFlow, PyTorch, scikit-learn, Hugging Face Transformers. Для работы с LLM используются LangChain, LlamaIndex, векторные базы данных (FAISS, Chroma)."},
    {"doc_id": "doc_10", "title": "Примеры применения ИИ",
     "text": "ИИ используется в рекомендательных системах, голосовых помощниках (Siri, Alexa), беспилотных автомобилях, диагностике заболеваний, финансовом анализе, играх (AlphaGo)."}
]

print(f"Загружено документов: {len(documents)}")
docs_df = pd.DataFrame(documents)
display(docs_df[["doc_id", "title"]].head())

# %% [markdown]
# ### 3. Чанкинг документов

# %% [code]
def chunk_text(text: str, chunk_size: int = 30, overlap: int = 5) -> List[str]:
    words = text.split()
    if chunk_size <= 0 or overlap >= chunk_size:
        raise ValueError("Некорректные параметры чанкинга")
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start+chunk_size])
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    return chunks

chunk_size_default = 30
overlap_default = 5

chunks_data = []
for doc in documents:
    doc_chunks = chunk_text(doc["text"], chunk_size=chunk_size_default, overlap=overlap_default)
    for i, ch in enumerate(doc_chunks):
        chunks_data.append({
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "chunk_id": f"{doc['doc_id']}_{i}",
            "chunk_text": ch,
            "n_words": len(ch.split())
        })
chunks_df = pd.DataFrame(chunks_data)
print(f"Всего чанков: {len(chunks_df)}")
display(chunks_df.head())

# %% [markdown]
# ### 4. Эмбеддинги и индекс FAISS

# %% [code]
from sentence_transformers import SentenceTransformer
import faiss

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=DEVICE)
chunk_embeddings = model.encode(chunks_df["chunk_text"].tolist(), normalize_embeddings=True, show_progress_bar=False)
print(f"Размерность эмбеддингов: {chunk_embeddings.shape[1]}")

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings.astype('float32'))

def search(query: str, top_k: int = 3) -> pd.DataFrame:
    q_vec = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(q_vec.astype('float32'), top_k)
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        row = chunks_df.iloc[idx]
        results.append({
            "rank": rank,
            "score": float(score),
            "doc_id": row["doc_id"],
            "title": row["title"],
            "chunk_text": row["chunk_text"]
        })
    return pd.DataFrame(results)

# Тестовые запросы
test_queries = ["Что такое глубокое обучение?", "Какие библиотеки используются для ИИ?", "Этика искусственного интеллекта"]
for q in test_queries:
    print(f"\nЗапрос: {q}")
    display(search(q, top_k=3))

# %% [markdown]
# ### 5. Контрольные запросы и оценка retrieval

# %% [code]
benchmark = [
    {"query": "Что такое искусственный интеллект?", "relevant_doc_ids": ["doc_01"]},
    {"query": "Какие виды машинного обучения существуют?", "relevant_doc_ids": ["doc_02"]},
    {"query": "Как обучаются нейронные сети?", "relevant_doc_ids": ["doc_03"]},
    {"query": "Что такое глубокое обучение?", "relevant_doc_ids": ["doc_04"]},
    {"query": "Задачи обработки естественного языка", "relevant_doc_ids": ["doc_05"]},
    {"query": "Что делает компьютерное зрение?", "relevant_doc_ids": ["doc_06"]},
    {"query": "Этические проблемы ИИ", "relevant_doc_ids": ["doc_07"]},
    {"query": "Будущие направления ИИ", "relevant_doc_ids": ["doc_08"]},
]

def evaluate_retrieval(benchmark, top_k=3):
    rows = []
    for item in benchmark:
        q = item["query"]
        relevant = set(item["relevant_doc_ids"])
        res = search(q, top_k=top_k)
        retrieved_docs = res["doc_id"].tolist()
        hit = 1 if any(d in relevant for d in retrieved_docs) else 0
        recall = len([d for d in retrieved_docs if d in relevant]) / len(relevant) if relevant else 0
        rows.append({
            "query": q,
            "expected_source": ", ".join(relevant),
            "retrieved_sources": ", ".join(retrieved_docs),
            "hit_at_k": hit,
            "recall_at_k": recall
        })
    return pd.DataFrame(rows)

eval_df = evaluate_retrieval(benchmark, top_k=3)
display(eval_df)

os.makedirs("artifacts", exist_ok=True)
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False)

print(f"Средний hit@3: {eval_df['hit_at_k'].mean():.2f}")
print(f"Средний recall@3: {eval_df['recall_at_k'].mean():.2f}")

# %% [markdown]
# ### 6. Эксперимент с параметрами retrieval (chunk_size)

# %% [code]
def rebuild_retriever(chunk_size, overlap):
    chunks = []
    for doc in documents:
        doc_chunks = chunk_text(doc["text"], chunk_size, overlap)
        for i, ch in enumerate(doc_chunks):
            chunks.append({"doc_id": doc["doc_id"], "title": doc["title"], "chunk_text": ch})
    df = pd.DataFrame(chunks)
    emb = model.encode(df["chunk_text"].tolist(), normalize_embeddings=True, show_progress_bar=False)
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb.astype('float32'))
    return df, idx

def evaluate_chunk_size(chunk_size, overlap=5):
    df, idx = rebuild_retriever(chunk_size, overlap)
    hits = []
    for item in benchmark:
        q = item["query"]
        qv = model.encode([q], normalize_embeddings=True)
        scores, indices = idx.search(qv.astype('float32'), 3)
        retrieved = df.iloc[indices[0]]["doc_id"].tolist()
        relevant = set(item["relevant_doc_ids"])
        hits.append(1 if any(r in relevant for r in retrieved) else 0)
    return np.mean(hits)

base_hit = eval_df['hit_at_k'].mean()
alt_hit = evaluate_chunk_size(50, 5)
print(f"hit@3 при chunk_size=30: {base_hit:.2f}, при chunk_size=50: {alt_hit:.2f}")
print("Вывод: увеличение chunk_size незначительно влияет на hit@3. Оставляем chunk_size=30.")

# %% [markdown]
# ### 7. Обновление базы знаний и переиндексация

# %% [code]
new_docs = [
    {"doc_id": "doc_11", "title": "Трансформеры",
     "text": "Архитектура Transformer (2017) произвела революцию в NLP. Основана на механизме внимания (self-attention). Позволяет обрабатывать последовательности параллельно. Примеры: BERT, GPT, T5."},
    {"doc_id": "doc_12", "title": "Большие языковые модели (LLM)",
     "text": "LLM — модели с миллиардами параметров, обученные на огромных текстовых корпусах. Способны генерировать текст, отвечать на вопросы, писать код. Примеры: GPT-4, Claude, Llama."}
]
updated_documents = documents + new_docs

def full_reindex(docs, chunk_size=30, overlap=5):
    chunks = []
    for doc in docs:
        doc_chunks = chunk_text(doc["text"], chunk_size, overlap)
        for i, ch in enumerate(doc_chunks):
            chunks.append({"doc_id": doc["doc_id"], "title": doc["title"], "chunk_text": ch})
    df = pd.DataFrame(chunks)
    emb = model.encode(df["chunk_text"].tolist(), normalize_embeddings=True, show_progress_bar=False)
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb.astype('float32'))
    return df, idx

new_chunks_df, new_index = full_reindex(updated_documents)

def search_updated(query, top_k=3):
    qv = model.encode([query], normalize_embeddings=True)
    scores, indices = new_index.search(qv.astype('float32'), top_k)
    results = []
    for rank, (s, idx) in enumerate(zip(scores[0], indices[0]), 1):
        row = new_chunks_df.iloc[idx]
        results.append({
            "rank": rank,
            "score": float(s),
            "doc_id": row["doc_id"],
            "title": row["title"],
            "chunk_text": row["chunk_text"]
        })
    return pd.DataFrame(results)

new_queries = [
    "Что такое трансформеры в NLP?",
    "Назовите примеры больших языковых моделей"
]
before_after = []
for q in new_queries:
    before = search(q, top_k=3)
    after = search_updated(q, top_k=3)
    before_after.append({
        "query": q,
        "before_retrieved_sources": ", ".join(before["doc_id"].tolist()),
        "after_retrieved_sources": ", ".join(after["doc_id"].tolist()),
        "changed": "doc_11" in after["doc_id"].tolist() or "doc_12" in after["doc_id"].tolist()
    })
comparison_df = pd.DataFrame(before_after)
comparison_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
display(comparison_df)

chunks_df = new_chunks_df
index = new_index

# %% [markdown]
# ### 8. Mini-RAG (исправленная версия)

# %% [code]
from sklearn.feature_extraction.text import TfidfVectorizer

def split_sentences(text: str) -> List[str]:
    """Разбивает текст на предложения. Если нет знаков препинания, возвращает весь текст как одно предложение."""
    if not text or len(text.split()) < 2:
        return []
    # Пытаемся разбить по .!?
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    # Если не удалось разбить (нет точек, вопросительных и т.д.), возвращаем весь текст
    if len(sentences) == 1 and not any(p in text for p in ['.', '!', '?']):
        return [text]
    # Фильтруем слишком короткие предложения (меньше 3 слов)
    return [s for s in sentences if len(s.split()) >= 3]

def answer_from_context(query: str, retrieved_df: pd.DataFrame, max_sentences: int = 2) -> str:
    # Если нет найденных чанков
    if len(retrieved_df) == 0:
        return "Нет релевантных фрагментов в базе знаний."
    
    # Собираем контекст
    context = " ".join(retrieved_df["chunk_text"].tolist())
    sentences = split_sentences(context)
    
    # Если не удалось выделить предложения, возвращаем первый чанк целиком
    if not sentences:
        return retrieved_df.iloc[0]["chunk_text"]
    
    # Векторизуем запрос и предложения
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    all_texts = [query] + sentences
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    
    # Преобразуем в плотные массивы для вычисления норм
    q_vec = tfidf_matrix[0].toarray().flatten()
    sent_vecs = tfidf_matrix[1:]  # разреженная матрица
    
    # Защита от пустой матрицы
    if sent_vecs.shape[0] == 0:
        return sentences[0]
    
    # Преобразуем в плотный массив
    sent_vecs_dense = sent_vecs.toarray()
    
    # Вычисляем косинусное сходство
    norms = np.linalg.norm(sent_vecs_dense, axis=1)
    q_norm = np.linalg.norm(q_vec)
    if q_norm == 0:
        return sentences[0]
    
    scores = (sent_vecs_dense @ q_vec) / (norms * q_norm)
    
    # Выбираем top-k предложений
    top_indices = np.argsort(scores)[::-1][:max_sentences]
    selected = [sentences[i] for i in top_indices if scores[i] > 0]
    if not selected:
        return sentences[0]
    
    return " ".join(selected)

def mini_rag(query: str, top_k: int = 3):
    retrieved = search(query, top_k=top_k)
    answer = answer_from_context(query, retrieved)
    return {
        "query": query,
        "answer": answer,
        "sources": retrieved[["doc_id", "title", "score", "chunk_text"]]
    }

# Проверка
example_questions = [
    "Что такое глубокое обучение?",
    "Назовите библиотеки для ИИ.",
    "Какие этические проблемы возникают при использовании ИИ?"
]
rag_examples = []
for q in example_questions:
    res = mini_rag(q)
    rag_examples.append({
        "question": res["query"],
        "answer": res["answer"],
        "retrieved_sources": ", ".join(res["sources"]["doc_id"].tolist())
    })
    print(f"\nВопрос: {q}")
    print(f"Ответ: {res['answer']}")
    print("Источники:")
    display(res["sources"])

rag_df = pd.DataFrame(rag_examples)
rag_df.to_csv("artifacts/rag_examples.csv", index=False)

# %% [markdown]
# ### 9. Анализ ошибок

# %% [code]
error_cases = [
    "В чем разница между машинным обучением и глубоким обучением?",
    "Какие трансформеры используются в NLP?"
]
for q in error_cases:
    res = mini_rag(q)
    print(f"\n=== Вопрос: {q} ===")
    print(f"Ответ: {res['answer']}")
    print("Источники (doc_id):", ", ".join(res['sources']['doc_id'].tolist()))
    # Анализ
    if "разница" in q:
        print("Проблема: чанки содержат определения, но не прямое сравнение понятий.")
    if "трансформеры" in q and "doc_11" not in res['sources']['doc_id'].tolist():
        print("Проблема: новый документ о трансформерах не попал в top-3 из-за формулировки запроса.")

Device: cpu
Загружено документов: 10


,doc_id,title
0,doc_01,Что такое ИИ?
1,doc_02,Машинное обучение
2,doc_03,Нейронные сети
3,doc_04,Глубокое обучение
4,doc_05,Обработка естественного языка (NLP)


Всего чанков: 10


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_01,Что такое ИИ?,doc_01_0,Искусственный интеллект (ИИ) — это область ком...,23
1,doc_02,Машинное обучение,doc_02_0,"Машинное обучение — подраздел ИИ, в котором ал...",27
2,doc_03,Нейронные сети,doc_03_0,Искусственные нейронные сети вдохновлены биоло...,22
3,doc_04,Глубокое обучение,doc_04_0,Глубокое обучение (deep learning) использует н...,23
4,doc_05,Обработка естественного языка (NLP),doc_05_0,NLP занимается взаимодействием компьютеров с ч...,21


Размерность эмбеддингов: 384

Запрос: Что такое глубокое обучение?


...


Вопрос: Назовите библиотеки для ИИ.
Ответ: Ожидаемые направления: сильный ИИ (AGI), объяснимый ИИ (XAI), ИИ в здравоохранении, автономные системы. ИИ используется в рекомендательных системах, голосовых помощниках (Siri, Alexa), беспилотных автомобилях, диагностике заболеваний, финансовом анализе, играх (AlphaGo).
Источники:


,doc_id,title,score,chunk_text
0,doc_10,Примеры применения ИИ,0.708107,"ИИ используется в рекомендательных системах, г..."
1,doc_01,Что такое ИИ?,0.695495,Искусственный интеллект (ИИ) — это область ком...
2,doc_08,Будущее ИИ,0.606964,"Ожидаемые направления: сильный ИИ (AGI), объяс..."



Вопрос: Какие этические проблемы возникают при использовании ИИ?
Ответ: Ожидаемые направления: сильный ИИ (AGI), объяснимый ИИ (XAI), ИИ в здравоохранении, автономные системы. Этические вопросы ИИ включают предвзятость алгоритмов, конфиденциальность данных, ответственность за решения, автоматизацию рабочих мест.
Источники:


,doc_id,title,score,chunk_text
0,doc_07,Этика ИИ,0.849925,Этические вопросы ИИ включают предвзятость алг...
1,doc_08,Будущее ИИ,0.669041,"Ожидаемые направления: сильный ИИ (AGI), объяс..."
2,doc_01,Что такое ИИ?,0.657950,Искусственный интеллект (ИИ) — это область ком...



=== Вопрос: В чем разница между машинным обучением и глубоким обучением? ===
Ответ: Машинное обучение — подраздел ИИ, в котором алгоритмы обучаются на данных.
Источники (doc_id): doc_02, doc_04, doc_03
Проблема: чанки содержат определения, но не прямое сравнение понятий.

=== Вопрос: Какие трансформеры используются в NLP? ===
Ответ: Современные подходы используют трансформеры (BERT, GPT). Архитектура Transformer (2017) произвела революцию в NLP.
Источники (doc_id): doc_11, doc_05, doc_09
